## Web Scraping Data Merek

In [ ]:
!pip install requests pandas tqdm -q

In [ ]:
import json
import requests
import pandas as pd
from tqdm import tqdm

In [ ]:
URL = "https://pdki-indonesia.dgip.go.id/search?advanced=1"

COOKIE = "PASTE_COOKIE_KAMU"

NEXT_ACTION = "87384f7de714212dd000e76fe09426dca6735260"

NEXT_ROUTER_STATE_TREE = "%5B%22%22%2C%7B%22children%22%3A%5B%22(with-header)%22%2C%7B%22children%22%3A%5B%22search%22%2C%7B%22children%22%3A%5B%22__PAGE__%22%2C%7B%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%2Ctrue%5D"

HEADERS = {
    "accept": "text/x-component",
    "content-type": "text/plain;charset=UTF-8",
    "origin": "https://pdki-indonesia.dgip.go.id",
    "referer": "https://pdki-indonesia.dgip.go.id/search?advanced=1",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
    "next-action": NEXT_ACTION,
    "next-router-state-tree": NEXT_ROUTER_STATE_TREE,
    "cookie": COOKIE
}

In [ ]:
def extract_json_objects(text, marker='{"_index":"trademark"'):

    import json

    objects = []

    start = 0

    while True:

        idx = text.find(marker, start)

        if idx == -1:
            break

        brace_count = 0
        in_string = False
        escape = False

        end = idx

        for i in range(idx, len(text)):

            ch = text[i]

            if escape:
                escape = False
                continue

            if ch == "\\":
                escape = True
                continue

            if ch == '"':
                in_string = not in_string
                continue

            if not in_string:

                if ch == "{":
                    brace_count += 1

                elif ch == "}":
                    brace_count -= 1

                    if brace_count == 0:
                        end = i + 1
                        break

        try:
            obj = json.loads(text[idx:end])
            objects.append(obj)

        except:
            pass

        start = end

    return objects

In [ ]:
def get_merek_per_kelas(kelas):

    payload = [
        {
            "index": "trademark",
            "body": {
                "from": 0,
                "size": 300,
                "query": {
                    "bool": {
                        "must": [
                            {
                                "match": {
                                    "t_class.class_no": str(kelas)
                                }
                            }
                        ],
                        "filter": [
                            {
                                "range": {
                                    "tanggal_permohonan": {
                                        "gte": "2016-01-01",
                                        "lte": "2026-12-31"
                                    }
                                }
                            }
                        ]
                    }
                }
            }
        }
    ]

    res = requests.post(
        URL,
        headers=HEADERS,
        data=json.dumps(payload)
    )

    if res.status_code != 200:
        print("Gagal kelas", kelas)
        return []

    objects = extract_json_objects(res.text)

    records = []

    for item in objects:

        src = item.get("_source", {})

        owner = src.get("owner", [])

        owner_name = None

        if owner:
            owner_name = owner[0].get("tm_owner_name")

        records.append({
            "id": src.get("id"),
            "kelas": kelas,
            "nama_merek": src.get("nama_merek"),
            "pemilik": owner_name,
            "nomor_permohonan": src.get("nomor_permohonan"),
            "tanggal_permohonan": src.get("tanggal_permohonan"),
            "status_permohonan": src.get("status_permohonan")
        })

    return records

In [ ]:
all_records = []

for kelas in tqdm(range(1, 46)):

    hasil = get_merek_per_kelas(kelas)

    print(
        f"Kelas {kelas} : {len(hasil)} merek"
    )

    all_records.extend(hasil)

  2%|▏         | 1/45 [00:16<12:03, 16.44s/it]

Kelas 1 : 300 merek


  4%|▍         | 2/45 [00:30<10:35, 14.78s/it]

Kelas 2 : 300 merek


  7%|▋         | 3/45 [00:41<09:22, 13.39s/it]

...

Kelas 43 : 300 merek


 98%|█████████▊| 44/45 [13:34<00:26, 26.91s/it]

Kelas 44 : 300 merek


100%|██████████| 45/45 [13:48<00:00, 18.41s/it]

Kelas 45 : 300 merek


In [ ]:
df = pd.DataFrame(all_records)

print(df.shape)

df.head()

(13500, 7)


,id,kelas,nama_merek,pemilik,nomor_permohonan,tanggal_permohonan,status_permohonan
0,IPT2020045216,1,REVO CLEAN,BRIAN CHRISTIAN,DID2020034848,2020-06-23,(TM) Dianggap Ditarik Kembali
1,IPT2020059001,1,Cireng Bawell,irvan sanjaya,DID2020039556,2020-07-20,(TM) Dianggap Ditarik Kembali
2,IPT2021002558,1,FIT WITH HONEY,UMMUL KHAIRIYAH,DID2021001376,2021-01-07,(TM) Dianggap Ditarik Kembali
3,IPT2021002932,1,TeKo,Universitas Nasional Karangturi,DID2021001591,2021-01-08,(TM) Dianggap Ditarik Kembali
4,IPT2021000478,1,zeolite.id,PT. Zeolite Natura Tangguh,DID2021000455,2021-01-04,(TM) Dianggap Ditarik Kembali


In [ ]:
df.to_csv(
    "dataset_merek_2016_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Selesai")

Selesai


In [ ]:
from google.colab import files

files.download(
    "dataset_merek_2016_2026.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>